# P2 — Vietnamese News Classification
## Phase 1 — Exploratory Data Analysis (EDA)

**Mục đích**: Khám phá dataset sau khi `prepare_data.py` chạy xong.

**Lưu ý**:
- KHÔNG fit TF-IDF trong notebook này.
- KHÔNG train model.
- Chỉ phân tích dữ liệu đã clean và split.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

with open(DATA_DIR / "categories.json", encoding="utf-8") as f:
    categories_meta = json.load(f)

with open(DATA_DIR / "dataset_metadata.json", encoding="utf-8") as f:
    dataset_meta = json.load(f)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Categories:", categories_meta["categories"])

## 1. Dataset Overview

In [ ]:
print("Total samples:", len(train_df) + len(val_df) + len(test_df))
print("Number of classes:", categories_meta["num_classes"])
print("Split sizes:", dataset_meta["split"])

## 2. Missing Values

In [ ]:
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"--- {name} ---")
    print(df.isna().sum())

## 3. Duplicate Count

In [ ]:
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name} duplicates on clean_text:", df["clean_text"].duplicated().sum())

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, [("Train", train_df), ("Val", val_df), ("Test", test_df)]):
    sns.countplot(data=df, y="label", ax=ax, order=categories_meta["categories"])
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. Text Length Distribution

In [ ]:
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    df["title_len"] = df["title"].str.len()
    df["content_len"] = df["content"].str.len()
    df["text_len"] = df["clean_text"].str.len()

train_df[["title_len", "content_len", "text_len"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["title_len", "content_len", "text_len"]):
    sns.histplot(train_df[col], bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 6. Samples Per Class

In [ ]:
summary = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int)
summary